# 01 -- Post-processing & Plotting (DCC process D1, downstream applications)

**SEA-FORWARD** OceanPrediction-A toolkit -- OPERA Capacity Development

SEA-FORWARD implements the **OceanPrediction-A** value chain end to end:

    Upstream Data (U) -> Core Forecasting Engine (C1) -> Verification & Analysis (V1) -> Downstream Applications (D1)

This notebook is **D1**: it takes CROCO output that Core process **C1**
already produced (`sftools`'s CROCO run) and turns it into the static
figures a downstream user actually looks at -- horizontal maps, vertical
sections/profiles, Hovmoller diagrams and time series for SST, SSH,
currents and salinity -- exactly the D1 deliverable SEA-FORWARD's own docs
describe: *"Jupyter notebooks for SST, SSH, currents, MLD and salinity,
with guided exercises"* (see the
[SEA-FORWARD documentation](https://sea-forward.readthedocs.io/en/latest/),
Phase 5 -- [Post-processing](https://sea-forward.readthedocs.io/en/latest/phase5/05_postprocessing/):
[Surface fields](https://sea-forward.readthedocs.io/en/latest/phase5/surface/),
[Dynamics](https://sea-forward.readthedocs.io/en/latest/phase5/dynamics/),
[Vertical structure](https://sea-forward.readthedocs.io/en/latest/phase5/vertical/),
[Through time](https://sea-forward.readthedocs.io/en/latest/phase5/time/)).

A D1 product is only as trustworthy as the C1 run it's drawn from --
SEA-FORWARD's own value chain puts **V1** (`02_validation.ipynb` / Step
4.2, RMSE/bias/spatial-correlation/Taylor-diagram skill against the
Copernicus Marine Forecast, satellite SST/SSS and in-situ observations)
*between* C1 and D1 for exactly this reason. This notebook doesn't re-run
that check -- it assumes the cycle you pick below has already passed (or
is being knowingly inspected pre-validation).

Sections below (merged from two earlier revisions of this notebook, one
richer in vertical/Hovmoller coverage, the other adding the closing
validation-comparison cell):

| Section | Content |
|---|---|
| 0 | Find and select a cycle |
| Horizontal section | SST, salinity, current speed + vectors, vorticity, combined eddy view |
| Vertical section | salinity, temperature, speed, u, v along a transect |
| Vertical profile | salinity, temperature, speed, u, v at a point |
| Hovmoller diagrams | salinity, temperature, speed, u, v -- time-latitude and time-depth |
| Time series | SST, temperature/salinity at depth, SSH, speed at depth |


In [ ]:
import importlib
import os, re
import sftools
import sftools.postprocess as pp
import sftools.plotting    as pl
import sftools.validation  as val
importlib.reload(sftools.validation)
importlib.reload(sftools.plotting)
import sftools.validation  as val
import sftools.plotting as pl


## 0. Find and select a cycle

Same discovery pattern as `02_validation.ipynb` Section 1: every run
lives in a cycle directory named `YYYYMMDD`, sibling to every other cycle
under `<MAIN_DIR>/<CONFIG>/`. `RUN_TYPE` distinguishes a forecast cycle
(`fcst/CROCO_FILES/croco_his.nc`) from a hindcast cycle
(`hcast/CROCO_FILES/croco_his.nc`) -- this notebook's original example
pointed at a hindcast run, so that's the default here; set it to `"fcst"`
to browse forecast cycles instead. Every available cycle under
`MAIN_DIR/CONFIG` is listed below; pick one with `CYCLE` (or the
`SEAFORWARD_CYCLE` environment variable).


In [ ]:
CONFIG   = os.environ.get("SEAFORWARD_CONFIG", "Canary_12")
RUN_TYPE = os.environ.get("SEAFORWARD_RUN_TYPE", "fcst")   # "fcst" or "hcast"
MAIN_DIR = os.path.expanduser(
    os.environ.get("SEAFORWARD_MAIN_DIR",
                   f"~/seaforward/{'forecast' if RUN_TYPE == 'fcst' else 'hindcast'}/model-runs"))

def _his_path(main_dir, config, cycle, run_type):
    return os.path.join(main_dir, config, cycle, run_type, "CROCO_FILES", "croco_his.nc")

def _list_cycles(main_dir, config, run_type):
    base = os.path.join(main_dir, config)
    if not os.path.isdir(base):
        return []
    return sorted(name for name in os.listdir(base)
                 if re.match(r"^\d{8}$", name)
                 and os.path.exists(_his_path(main_dir, config, name, run_type)))

AVAILABLE_CYCLES = _list_cycles(MAIN_DIR, CONFIG, RUN_TYPE)
print(f"{RUN_TYPE} cycles found under {os.path.join(MAIN_DIR, CONFIG)}: {AVAILABLE_CYCLES}")

# >>> SET THIS to the cycle you want to open, e.g. "20251225" <<<
CYCLE = os.environ.get("SEAFORWARD_CYCLE", "20251225")

# hindcast runs from the GLORYS reanalysis (CF time origin year 1993);
# forecast runs from the Copernicus Marine Forecast / Mercator anfc (2000)
YORIG = 1993 if RUN_TYPE == "hcast" else 2000

H = _his_path(MAIN_DIR, CONFIG, CYCLE, RUN_TYPE)
if not os.path.exists(H):
    raise FileNotFoundError(
        f"No CROCO history file for cycle {CYCLE!r} at {H}.\n"
        f"Available {RUN_TYPE} cycles under {os.path.join(MAIN_DIR, CONFIG)}: "
        f"{AVAILABLE_CYCLES if AVAILABLE_CYCLES else '(none found)'}")

print(f"Opening {RUN_TYPE} cycle {CYCLE}")
print(f"  CROCO history: {H}")


## Open data

In [ ]:
ds = pp.open_history(H, Yorig=YORIG)

#### Parameters

In [ ]:
depth = 1000
lon0, lat0, lon1, lat1 = -21, -16, 21, 21
isobaths = [100, 200, 500, 1000, 2000]


## Horizontal section

### Salinity

In [ ]:
%matplotlib inline

fig = pl.plot(pp.field(ds, "salt", depth_m=depth),ds=ds, isobaths=isobaths)   # no out= -> returns the figure


### Temperature

In [ ]:
fig = pl.plot(pp.field(ds, "temp", depth_m=depth),ds=ds, isobaths=isobaths)   # no out= -> returns the figure
fig

### Curent & Speed at depth

In [ ]:
# u, v = pp.surface_uv(ds,tindex=-1); # u,v at surface
# u, v = pp.rotate_uv(ds, u, v) #rotate to east/north for surface currents
u, v = u, v = pp.uv_at_depth(ds, depth_m=depth, tindex=-1, rotate=True)  # u,v at depth; 
fig=pl.plot(pp.speed_map(ds,depth_m=depth),ds=ds, uv=(u, v),isobaths=isobaths,uv_scale=4, uv_skip=3, uv_ref=0.2)   # saves to file, returns the filename

### Vorticity

In [ ]:
fig=pl.plot(pp.vorticity(ds,depth_m=depth, normalized=True))
fig

### Combined eddy view: shaded base field + vorticity contours or current vectors

In [ ]:
# overlay : ('vort', vort_da)  -> vorticity contours (from vort/f)
#               ('uv', (u, v))     -> current vectors
fig = pl.plot_eddy(pp.field(ds, "temp", depth_m=depth),ds=ds, isobaths=isobaths,overlay= ('uv', (u, v)),uv_scale=4, uv_skip=3, uv_ref=0.2)   # no out= -> returns the figure

In [ ]:
# overlay : ('vort', vort_da)  -> vorticity contours (from vort/f)
#               ('uv', (u, v))     -> current vectors
vort_da = pp.vorticity(ds,depth_m=depth, normalized=True)
fig = pl.plot_eddy(pp.field(ds, "temp", depth_m=depth),ds=ds, isobaths=isobaths,overlay= ('vort', vort_da))   # no out= -> returns the figure

# ________________________________________________________________________________________________________________________________________

## Vertical section

### salinity

In [ ]:
fig=pl.plot(pp.section(ds, "salt",  lon0, lon1, lat0, lat1),contour_colors="white", levels=10)   # no out= -> returns the figure
fig

### temperature

In [ ]:
fig=pl.plot(pp.section(ds, "temp",  lon0, lon1, lat0, lat1),contour_colors="white", levels=10)   # no out= -> returns the figure
fig

### speed

In [ ]:
fig=pl.plot(pp.section(ds, "speed",  lon0, lon1, lat0, lat1),contour_colors="white", levels=5)   # no out= -> returns the figure
fig

### Zonal current (u)

In [ ]:
fig=pl.plot(pp.section(ds, "u",  lon0, lon1, lat0, lat1),contour_colors="black", levels=5)   # no out= -> returns the figure

### Meridional current (v)

In [ ]:
fig=pl.plot(pp.section(ds, "v",  lon0, lon1, lat0, lat1),contour_colors="black", levels=5)   # no out= -> returns the figure

# ________________________________________________________________________________________________________________________________________

## Vertcal Profil

###  salinity

In [ ]:
fig=pl.plot(pp.profile(ds, "salt",  lon0, lat0))

###  temperature

In [ ]:
fig=pl.plot(pp.profile(ds, "temp",  lon0, lat0))
fig

###  speed

In [ ]:
fig=pl.plot(pp.profile(ds, "speed",  lon0, lat0))
fig

### Zonal current (u)

In [ ]:
fig = pl.plot(pp.profile(ds, "u", lon0, lat0))

### Meridional current (v)

In [ ]:
fig = pl.plot(pp.profile(ds, "v", lon0, lat0))

## Hovmöller diagrams

### salinity

In [ ]:
fig = pl.plot(pp.hovmoller(ds, "salt", "time_lat", lon0=-19))
fig

In [ ]:
fig = pl.plot(pp.hovmoller(ds, "salt", kind="time_depth", lon0=lon0, lat0=lat0))


### temperature

In [ ]:
fig = pl.plot(pp.hovmoller(ds, "temp", "time_lat", lon0=-19))
fig

In [ ]:
fig = pl.plot(pp.hovmoller(ds, "temp", kind="time_depth", lon0=lon0, lat0=lat0))


### Speed

In [ ]:
fig = pl.plot(pp.hovmoller(ds, "speed", kind="time_depth", lon0=lon0, lat0=lat0))


### Zonal current (u)

In [ ]:
fig = pl.plot(pp.hovmoller(ds, "u", kind="time_depth", lon0=lon0, lat0=lat0))


### Meridional current (v)

In [ ]:
fig = pl.plot(pp.hovmoller(ds, "v", kind="time_depth", lon0=lon0, lat0=lat0))


## Time series

#### surface SST over time

In [ ]:
fig=pl.plot(pp.timeseries(ds, "temp",   lon0=-19, lat0=21))
fig

### temperature at depth over time

In [ ]:
fig=pl.plot(pp.timeseries(ds, "temp", lon0=-19, lat0=21,depth_m=50))
fig

### salinity at depth over time

In [ ]:
fig=pl.plot(pp.timeseries(ds, "salt", lon0=-19, lat0=21,depth_m=50))
fig

#### SSH over time

In [ ]:
fig = pl.plot(pp.timeseries(ds, "zeta",  lon0=-19, lat0=21))
fig

#### Speed at depth over time

In [ ]:
fig = pl.plot(pp.timeseries(ds, "speed",  lon0=-19, lat0=21, depth_m=50))


# _________________________________________________________________________________________________________________________________________________________________________